# Evaluate diffusion samples on CPU

The evaluator in `diffusion/src/evaluation/` uses JAXVacua and SciPy directly. Samples contain integer fluxes and full complex physical initializers; no discretization factor or transformer evaluation imports are used. The sample JSON sidecar supplies the geometry panel, mode, and conditioning target.

Choose **A. Local Jupyter** or **B. Google Colab**, skip the other route, then run **C. Shared evaluation settings** and the desired evaluation blocks. Do not use “Run All” across both setup routes. Install the CPU dependencies in your selected environment before evaluating. Older integer-encoded samples must be regenerated with the updated sampler.

## A. Local Jupyter setup

Select a **Python 3.12 or newer** kernel. The pinned JAXVacua package cannot be installed in Python 3.11, even if that environment already has JAX. Sampling and evaluation may use different kernels; the saved sample files connect them.

Run this block locally and skip B. Start Jupyter from the repository root or `diffusion/`, or set `REPO_ROOT` explicitly. Samples and results default to `diffusion/samples` and `diffusion/evaluations`. The printed Python path identifies the environment that needs the evaluation dependencies.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = None  # Or Path("/absolute/path/to/towards_type_IIB_foundation_model").
if REPO_ROOT is None:
    REPO_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
                      if (path / "diffusion/src/evaluation/evaluate.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT to your local repository path.")
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
SAMPLES_DIR = REPO_ROOT / "diffusion/samples"
RESULTS_DIR = REPO_ROOT / "diffusion/evaluations"
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
if sys.version_info < (3, 12):
    raise RuntimeError("Evaluation requires Python >= 3.12 for the pinned JAXVacua package. "
                       "Select a Python 3.12+ kernel, then rerun local setup.")


### Install local evaluation dependencies (once per environment)

Skip this block if the selected kernel already has the dependencies. Otherwise, run it after local setup; it installs the repository's CPU requirements into the **active kernel's Python**, avoiding a different terminal environment. Restart the kernel after installation, rerun local setup, then go to C. JAXVacua is pinned to the same revision as the repository requirements.


In [ ]:
import subprocess

if "REPO_ROOT" not in globals():
    raise RuntimeError("Run local Jupyter setup first.")
if sys.version_info < (3, 12):
    raise RuntimeError("Select a Python 3.12+ kernel before installing evaluation dependencies.")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements-cpu.txt"),
])
print("Installation complete. Restart the kernel, rerun local setup, then run C.")


## B. Google Colab setup

Skip this section locally. Select a CPU runtime and run the installation block once. Restart the kernel if requested, then run the Drive block. Evaluation does not need a GPU.

In [ ]:
%cd /content
!apt-get install git-lfs
!git lfs install
!git clone https://github.com/StringJAX/towards_type_IIB_foundation_model.git
%cd towards_type_IIB_foundation_model
%pip install -r requirements-cpu.txt
%cd diffusion

### Colab paths and Drive

Mount the same Drive folder used by the sampling notebook. Both the sample pickle and its JSON sidecar must be present. Evaluation results are saved alongside the samples in an `evaluations` subdirectory.

In [ ]:
from pathlib import Path
from google.colab import drive

REPO_ROOT = Path("/content/towards_type_IIB_foundation_model")
drive.mount("/content/drive")
SAMPLES_DIR = Path("/content/drive/MyDrive/IIB_foundation_model_samples")
RESULTS_DIR = SAMPLES_DIR / "evaluations"

## C. Shared evaluation settings

Run this block after either setup route, before importing JAX in this kernel. It first checks the Python version and required packages in the active kernel. It reloads the diffusion sample loader, physics helpers, and evaluator before importing `evaluate_samples`, so rerunning this block picks up source changes in an existing kernel. It prints the source path and expected sample format. If another checkout is already imported, restart the kernel and select the intended repository. `TOL` sets the maximum F-term residual. The default solver is SciPy's `hybr`; set `USE_NEWTON=True` to use JAXVacua's Newton solver. Set `VERBOSE=True` to print successful row residuals.

Each example below reads its mode, physical target, and selected geometries from the sidecar. Change the filename to evaluate a pre-trained model or another saved run. Existing result files are protected against overwriting; choose a new `RESULTS_DIR` when repeating an evaluation.

In [ ]:
import importlib
import importlib.util
import os
import sys

if "REPO_ROOT" not in globals():
    raise RuntimeError("Run local Jupyter or Colab setup first.")
if sys.version_info < (3, 12):
    raise RuntimeError("Select a Python 3.12+ kernel for JAXVacua evaluation.")
missing = [name for name in ("jax", "jaxvacua", "scipy")
           if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        f"Missing evaluation dependencies: {', '.join(missing)}. Kernel: {sys.executable}. "
        "Run the local dependency installation block (or Colab installation), "
        "restart the kernel, then rerun setup."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("XLA_FLAGS", "--xla_backend_optimization_level=2")
# Reload dependencies first: a repeated plain import keeps cached functions.
importlib.invalidate_caches()
for module_name in (
    "diffusion.src.evaluation.samples",
    "diffusion.src.evaluation.physics",
    "diffusion.src.evaluation",
    "diffusion.src.evaluation.evaluate",
):
    module = importlib.import_module(module_name)
    if not Path(module.__file__).resolve().is_relative_to(REPO_ROOT):
        raise RuntimeError(f"{module_name} was imported from {module.__file__}. "
                           "Restart the kernel and select the intended repository.")
    importlib.reload(module)
from diffusion.src.evaluation.evaluate import evaluate_samples
from diffusion.src.evaluation.samples import SAMPLE_FORMAT
print(f"Evaluator: {module.__file__}; sample format: {SAMPLE_FORMAT}")

TOL = 1e-10
USE_NEWTON = False
VERBOSE = False
EVALUATION_OPTIONS = {
    "output_dir": RESULTS_DIR,
    "tol": TOL,
    "use_newton": USE_NEWTON,
    "verbose": VERBOSE,
}

## All-geometry model

Evaluates the 20 prompted geometries. Use `uncon_pretrained` in the filename for the pre-trained model.

In [ ]:
filename = "diffusion_uncon_grpo_samples.p"
result_file = evaluate_samples(samples_file=SAMPLES_DIR / filename, **EVALUATION_OPTIONS)

## Fine-tuned model at $h^{1,2}=5$

Evaluates the five recorded $h^{1,2}=5$ geometries.

In [ ]:
filename = "diffusion_h12_5_finetuned_grpo_samples.p"
result_file = evaluate_samples(samples_file=SAMPLES_DIR / filename, **EVALUATION_OPTIONS)

## Scratch benchmark at $h^{1,2}=5$

Use `h12_5_scratch_pretrain` in the filename for the model before GRPO.

In [ ]:
filename = "diffusion_h12_5_scratch_grpo_samples.p"
result_file = evaluate_samples(samples_file=SAMPLES_DIR / filename, **EVALUATION_OPTIONS)

## $N_{\rm flux}$ conditioning

Reads the requested integer target from the sidecar and measures the tadpole of each unique valid solution.

In [ ]:
filename = "diffusion_nflux_grpo_samples.p"
result_file = evaluate_samples(samples_file=SAMPLES_DIR / filename, **EVALUATION_OPTIONS)

## $|W_0|$ conditioning

Reads the target in physical units and measures the gauge-invariant superpotential magnitude.

In [ ]:
filename = "diffusion_w0_grpo_samples.p"
result_file = evaluate_samples(samples_file=SAMPLES_DIR / filename, **EVALUATION_OPTIONS)

## $1/g_s$ conditioning

Uses the saved complex tau prediction as the initializer. The resulting distribution is Im(tau); the prompt does not overwrite the initializer.

In [ ]:
filename = "diffusion_gs_grpo_samples.p"
result_file = evaluate_samples(samples_file=SAMPLES_DIR / filename, **EVALUATION_OPTIONS)

## Optional reference-data validation check

This block loads 500 flux vectors from `transformer/data/data_multiple`, balanced across $h^{1,2}=2,3,4,5$, and validates them using the same diffusion physics code. These files contain neither geometry labels nor moduli: the check tries candidate geometries with matching $h^{1,2}$ using the geometry-log mean initializers. An unresolved row may need a better initial guess; it is not certified invalid.

The JSON report records every selected row index, successful geometry, residual, and per-stage counts. This is a separate, longer validation check, not a required step for evaluating model samples. Choose a fresh report filename if it already exists.

In [ ]:
from diffusion.src.evaluation.validate_dataset import validate_dataset

report_file = validate_dataset(
    num_samples=500,
    data_dir=REPO_ROOT / "transformer/data/data_multiple",
    output=RESULTS_DIR / "reference_data_validation_500.json",
    tol=TOL,
)

## Saved results

Conditional `*_valid.p` files hold `{prompt: {"dist": ..., "validity": ...}}`, with physical target values in the prompt keys. Unconditional files contain per-geometry counts. Each summary records sample counts, validity, unique-valid rates, solver settings, and sampling metadata.

Command-line equivalent from the repository root:

```bash
python -m diffusion.src.evaluation.evaluate --samples_file diffusion_nflux_grpo_samples.p
```